In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'
import pandas as pd
import spacy
import re
import time
import numpy as np
import time

import ast
from ast import literal_eval

In [ ]:
!python -m spacy download en_core_web_lg

2023-08-04 11:39:44.049012: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.7/587.7 MB 2.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')


In [ ]:
!python -m spacy download de_core_news_sm

2023-08-04 11:40:17.778053: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 12.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')


In [ ]:
from spacy.matcher import PhraseMatcher
from spacy.matcher import Matcher
import re

In [ ]:
# Model data
model = {'en': spacy.load("en_core_web_lg"), 'de': spacy.load("de_core_news_sm")}

In [ ]:
#function to split string in the row by " | "
def split_form(row):
    row_split = row.split(" | ")
    return row_split

In [ ]:
# function to add ... for inspiration alternatives
def add_dots(row):
    list_out =[]
    if type(row)==str:
        row_list = row.split(" | ")
        for item in row_list:
            res = re.search(r"\...", item)
            if res is not None:
                list_out.append(item)
            else:
                list_out.append(item+"(...)")
        return " | ".join(list_out)

In [ ]:
#function to clean alternatives from pipe sign
def clean_str(row):
    array_out =[]
    if len(row)==0:
        return list()
    else:
        array_out =[i.strip(' |') for i in row]
        new_list = [x for x in array_out if x != '']
        return new_list

In [ ]:
#function to separate one term binary pronouns excluding / (for example, exclude he/she)
def separate_binary_pronouns(df):
    list_word = []
    list_sentence = []

    list_word_type_word = []
    list_word_type_sentence = []

    list_category_word = []
    list_category_sentence = []

    list_primary_subcategory_word = []
    list_primary_subcategory_sentence = []

    list_sg_all_word = []
    list_sg_all_sentence = []

    list_pl_word = []
    list_pl_sentence = []

    list_pl_all_word = []
    list_pl_all_sentence = []

    list_alternatives_word = []
    list_alternatives_sentence = []


    for index, row in df.iterrows():
        word = row["Lemma"]
        span = re.search("/", word)

        if len(word.rsplit(" "))==1 and len(word.rsplit("/"))==1:
            list_word.append(word)
            list_word_type_word.append(row["Word_Type"])
            list_category_word.append(row["Category"])
            list_primary_subcategory_word.append(row["Primary_subcategory"])
            list_sg_all_word.append(row["Sg_all_split"])
            list_pl_word.append(row["Pl_all"])
            list_pl_all_word.append(row["Pl_all_split"])
            list_alternatives_word.append(row["Alt_split"])

    df_word = pd.DataFrame(list(zip(list_word, list_word_type_word, list_category_word, list_primary_subcategory_word, list_sg_all_word, list_pl_word, list_pl_all_word, list_alternatives_word)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split"])

    return df_word




In [ ]:
#function to split words and sentences into two different dataframes for singular/plural dataframes. Used for gendered category to handle multiterms pronouns
def split_binary_pronouns(df):
    list_word = []
    list_sentence = []

    list_word_type_word = []
    list_word_type_sentence = []

    list_category_word = []
    list_category_sentence = []

    list_primary_subcategory_word = []
    list_primary_subcategory_sentence = []

    list_secondary_subcategory_word = []
    list_secondary_subcategory_sentence = []

    list_sg_all_word = []
    list_sg_all_sentence = []

    list_pl_word = []
    list_pl_sentence = []

    list_pl_all_word = []
    list_pl_all_sentence = []

    list_alternatives_word = []
    list_alternatives_sentence = []


    for index, row in df.iterrows():
        word = row["Lemma"]
        span = re.search("/", word)


        if len(word.rsplit(" "))==1 and len(word.rsplit("/"))==1:
            list_word.append(word)
            list_word_type_word.append(row["Word_Type"])
            list_category_word.append(row["Category"])
            list_primary_subcategory_word.append(row["Primary_subcategory"])
            list_secondary_subcategory_word.append(row["Secondary_subcategory"])
            list_sg_all_word.append(row["Sg_all_split"])
            list_pl_word.append(row["Pl_all"])
            list_pl_all_word.append(row["Pl_all_split"])
            list_alternatives_word.append(row["Alt_split"])

        elif type(span) == re.Match and len(word.rsplit("/"))==2:
            list_sentence.append(word)
            list_word_type_sentence.append(row["Word_Type"])
            list_category_sentence.append(row["Category"])
            list_primary_subcategory_sentence.append(row["Primary_subcategory"])
            list_secondary_subcategory_sentence.append(row["Secondary_subcategory"])
            list_sg_all_sentence.append(row["Sg_all_split"])
            list_pl_sentence.append(row["Pl_all"])
            list_pl_all_sentence.append(row["Pl_all_split"])
            list_alternatives_sentence.append(row["Alt_split"])

        else:
            list_sentence.append(word)
            list_word_type_sentence.append(row["Word_Type"])
            list_category_sentence.append(row["Category"])
            list_primary_subcategory_sentence.append(row["Primary_subcategory"])
            list_secondary_subcategory_sentence.append(row["Secondary_subcategory"])
            list_sg_all_sentence.append(row["Sg_all_split"])
            list_pl_sentence.append(row["Pl_all"])
            list_pl_all_sentence.append(row["Pl_all_split"])
            list_alternatives_sentence.append(row["Alt_split"])

    df_word = pd.DataFrame(list(zip(list_word, list_word_type_word, list_category_word, list_primary_subcategory_word, list_secondary_subcategory_word, list_sg_all_word, list_pl_word, list_pl_all_word, list_alternatives_word)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory","Secondary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split"])
    df_sentence = pd.DataFrame(list(zip(list_sentence, list_word_type_sentence, list_category_sentence, list_primary_subcategory_sentence, list_secondary_subcategory_sentence, list_sg_all_sentence, list_pl_sentence, list_pl_all_sentence, list_alternatives_sentence)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Secondary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split"])

    return df_word, df_sentence




In [ ]:
#function to clean alternatives from pipe sign
def replace_tolist(file_csv):

    # creating a variable and storing the search text
    search_text = "['']"

    # creating a variable and storing the replace text
    replace_text = "[]"

    # Opening our text file in read only
    # mode using the open() function
    with open(file_csv, 'r') as file:

        # Reading the content of the file
        # using the read() function and storing
        # them in a new variable
        data = file.read()

        # Searching and replacing the text
        # using the replace() function
        data = data.replace(search_text, replace_text)

    with open(file_csv, 'w') as file:
        # Writing the replaced data in our
        # text file
        file.write(data)
    return file_csv



In [ ]:
# use if you run scripts on google colabs
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# read csv file
df_rules = pd.read_csv("/content/drive/MyDrive/rules_04AUG_English.csv")

In [ ]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2287 entries, 0 to 2286
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Lemma                   2286 non-null   object 
 1   Word_Type               2287 non-null   object 
 2   Priority                0 non-null      float64
 3   Category                2287 non-null   object 
 4   Primary_subcategory     2287 non-null   object 
 5   Secondary_subcategory   671 non-null    object 
 6   Alt_Field               775 non-null    object 
 7   Alt_Sg_Replacement      1296 non-null   object 
 8   Identity_first          56 non-null     object 
 9   Alt_Sg_people_first     199 non-null    object 
 10  Alt_Sg_reframed         322 non-null    object 
 11  Alt_Pl_collective_noun  58 non-null     object 
 12  Alt_Pl                  395 non-null    object 
 13  Identity_first_pl       25 non-null     object 
 14  Alt_Pl_people_first     129 non-null    

In [ ]:
df_rules[df_rules["Lemma"].isna()]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Identity_first,Alt_Sg_people_first,...,Alt_Pl_people_first,Alt_Pl_reframed,3rd_party_alternatives,False_Positives,3rd_party_source,Sample_Sentences,Notes,ToClarify,Generated Examples,Unnamed: 23
1147,NaN,=,NaN,style,advanced_abbreviation,general,NaN,not applicable | not available,NaN,NaN,...,NaN,NaN,NaN,(N/A),NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
#convert NaN to string "N/A" in lemmas
df_rules.Lemma.fillna("N/A", inplace=True)

In [ ]:
df_rules.iloc[[246]]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Identity_first,Alt_Sg_people_first,...,Alt_Pl_people_first,Alt_Pl_reframed,3rd_party_alternatives,False_Positives,3rd_party_source,Sample_Sentences,Notes,ToClarify,Generated Examples,Unnamed: 23
246,manageress,s,NaN,gendered,titles,generic_plural,management,manager | they,NaN,NaN,...,NaN,people in a managing role | those who manage p...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2287 entries, 0 to 2286
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Lemma                   2287 non-null   object 
 1   Word_Type               2287 non-null   object 
 2   Priority                0 non-null      float64
 3   Category                2287 non-null   object 
 4   Primary_subcategory     2287 non-null   object 
 5   Secondary_subcategory   671 non-null    object 
 6   Alt_Field               775 non-null    object 
 7   Alt_Sg_Replacement      1296 non-null   object 
 8   Identity_first          56 non-null     object 
 9   Alt_Sg_people_first     199 non-null    object 
 10  Alt_Sg_reframed         322 non-null    object 
 11  Alt_Pl_collective_noun  58 non-null     object 
 12  Alt_Pl                  395 non-null    object 
 13  Identity_first_pl       25 non-null     object 
 14  Alt_Pl_people_first     129 non-null    

### Drop duplicates

In [ ]:
df_rules = df_rules.drop_duplicates()

In [ ]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2287 entries, 0 to 2286
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Lemma                   2287 non-null   object 
 1   Word_Type               2287 non-null   object 
 2   Priority                0 non-null      float64
 3   Category                2287 non-null   object 
 4   Primary_subcategory     2287 non-null   object 
 5   Secondary_subcategory   671 non-null    object 
 6   Alt_Field               775 non-null    object 
 7   Alt_Sg_Replacement      1296 non-null   object 
 8   Identity_first          56 non-null     object 
 9   Alt_Sg_people_first     199 non-null    object 
 10  Alt_Sg_reframed         322 non-null    object 
 11  Alt_Pl_collective_noun  58 non-null     object 
 12  Alt_Pl                  395 non-null    object 
 13  Identity_first_pl       25 non-null     object 
 14  Alt_Pl_people_first     129 non-null    

In [ ]:
#check missing values in Primary subcategory
df_rules[df_rules['Word_Type'].isna()]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Identity_first,Alt_Sg_people_first,...,Alt_Pl_people_first,Alt_Pl_reframed,3rd_party_alternatives,False_Positives,3rd_party_source,Sample_Sentences,Notes,ToClarify,Generated Examples,Unnamed: 23


In [ ]:
#check missing values in Primary subcategory
df_rules[df_rules['Category'].isna()]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Identity_first,Alt_Sg_people_first,...,Alt_Pl_people_first,Alt_Pl_reframed,3rd_party_alternatives,False_Positives,3rd_party_source,Sample_Sentences,Notes,ToClarify,Generated Examples,Unnamed: 23


In [ ]:
# drop words with missing category
df_rules.dropna(subset=['Category'], inplace=True)

In [ ]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2287 entries, 0 to 2286
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Lemma                   2287 non-null   object 
 1   Word_Type               2287 non-null   object 
 2   Priority                0 non-null      float64
 3   Category                2287 non-null   object 
 4   Primary_subcategory     2287 non-null   object 
 5   Secondary_subcategory   671 non-null    object 
 6   Alt_Field               775 non-null    object 
 7   Alt_Sg_Replacement      1296 non-null   object 
 8   Identity_first          56 non-null     object 
 9   Alt_Sg_people_first     199 non-null    object 
 10  Alt_Sg_reframed         322 non-null    object 
 11  Alt_Pl_collective_noun  58 non-null     object 
 12  Alt_Pl                  395 non-null    object 
 13  Identity_first_pl       25 non-null     object 
 14  Alt_Pl_people_first     129 non-null    

### Check missing values for word type

In [ ]:
#check missing values in Primary subcategory
df_rules[df_rules['Word_Type'].isna()]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Identity_first,Alt_Sg_people_first,...,Alt_Pl_people_first,Alt_Pl_reframed,3rd_party_alternatives,False_Positives,3rd_party_source,Sample_Sentences,Notes,ToClarify,Generated Examples,Unnamed: 23


### Check if subcategory name is missing and set the parent category name to subcategory if the subategory name is missing

In [ ]:
#check missing values in Primary subcategory
df_rules[df_rules['Primary_subcategory'].isna()]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Identity_first,Alt_Sg_people_first,...,Alt_Pl_people_first,Alt_Pl_reframed,3rd_party_alternatives,False_Positives,3rd_party_source,Sample_Sentences,Notes,ToClarify,Generated Examples,Unnamed: 23


In [ ]:
#fill the missing values in "primary subcategory" by values from "Category"
df_rules.Primary_subcategory.fillna(df_rules.Category, inplace=True)

In [ ]:
#df_rules[df_rules["Primary_subcategory"]=="Nazi_language"]

In [ ]:
df_rules["Primary_subcategory"].unique()

array(['advanced_ability', 'advanced_hidden_image', 'binary_pronouns',
       'female_stereotype', 'function', 'leadership', 'male_stereotype',
       'titles', 'communal', 'd_and_i', 'emotional_security', 'ableism',
       'antimuslim', 'antisemitism', 'homophobia', 'offensive_language',
       'racism', 'sexism', 'transphobia', 'xenophobia',
       'advanced_abbreviation', 'advanced_formality', 'advanced_hollow',
       'exaggerating', 'filler', 'age_old', 'age_young', 'agentic',
       'behavior', 'belief', 'classism', 'cognitive_ability',
       'cognitive_perception', 'color', 'culture', 'gender_identity',
       'hearing', 'learning', 'medical_state', 'mental_wellbeing',
       'migration', 'military_source', 'mobility', 'nazi_language',
       'racist_source', 'sexual_orientation', 'speech', 'sports_terms',
       'vision'], dtype=object)

In [ ]:
#df_rules = df_rules[df_rules["Primary_subcategory"]!="nazi_language"]

In [ ]:
#check if the values are filled
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2287 entries, 0 to 2286
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Lemma                   2287 non-null   object 
 1   Word_Type               2287 non-null   object 
 2   Priority                0 non-null      float64
 3   Category                2287 non-null   object 
 4   Primary_subcategory     2287 non-null   object 
 5   Secondary_subcategory   671 non-null    object 
 6   Alt_Field               775 non-null    object 
 7   Alt_Sg_Replacement      1296 non-null   object 
 8   Identity_first          56 non-null     object 
 9   Alt_Sg_people_first     199 non-null    object 
 10  Alt_Sg_reframed         322 non-null    object 
 11  Alt_Pl_collective_noun  58 non-null     object 
 12  Alt_Pl                  395 non-null    object 
 13  Identity_first_pl       25 non-null     object 
 14  Alt_Pl_people_first     129 non-null    

In [ ]:
# Check Identity_first alternatives
df_rules[df_rules['Identity_first'].notna()]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Identity_first,Alt_Sg_people_first,...,Alt_Pl_people_first,Alt_Pl_reframed,3rd_party_alternatives,False_Positives,3rd_party_source,Sample_Sentences,Notes,ToClarify,Generated Examples,Unnamed: 23
1491,Asperger,~s,NaN,unconscious_bias,advanced_ability,NaN,"--- only if the person prefers the term, or wh...",on the autism spectrum,Autistic | neurodivergent,person who has autism | someone with autism,...,NaN,NaN,NaN,NaN,DCC,My son has Asperger's and needs more support i...,Various related disabilities have been relabel...,https://medium.com/@MooniWolfKomoki/why-the-re...,NaN,NaN
1492,Asperger’s Syndrome,~s|||~s,NaN,unconscious_bias,advanced_ability,NaN,"--- only if the person prefers the term, or wh...",being on the autism spectrum,Autistic | neurodivergent,person who has autism | someone with autism,...,NaN,NaN,NaN,NaN,DCC,Someone with Asperger's Syndrome are often not...,Various related disabilities have been relabel...,https://molecularautism.biomedcentral.com/arti...,NaN,NaN
1494,birth-defect,s,NaN,unconscious_bias,advanced_ability,NaN,---name congenital health condition or disability,has a disability | congenital disability | has...,is disabled,NaN,...,NaN,NaN,NaN,NaN,DCC,"Birth defects can be diagnosed before, at, and...",Disability is preferred because of its more in...,https://www.lawinsider.com/dictionary/congenit...,NaN,NaN
1497,clumsy,a,NaN,unconscious_bias,advanced_ability,NaN,--- name specific disability,lacking coordination | developmental coordination,uncoordinated,NaN,...,NaN,NaN,NaN,NaN,DCC,Dyspraxia is sometimes called Clumsy Child Syn...,Portraying qualities of disabilities in defici...,https://themighty.com/topic/cerebral-palsy/con...,NaN,NaN
1498,coexisting difficulty,~a|s,NaN,unconscious_bias,advanced_ability,NaN,NaN,co-occurring disability | concurrent disabilit...,disabled,NaN,...,NaN,NaN,NaN,NaN,DCC,Poor motor coordination and cognition are coex...,Disability is preferred because of its more in...,https://academic.oup.com/jdsde/article/19/2/18...,NaN,NaN
1528,special ed,~a|s,NaN,unconscious_bias,advanced_ability,NaN,NaN,adaptive education | student in adaptive educa...,student,NaN,...,NaN,NaN,NaN,NaN,DCC,The school had a special ed class comprising o...,Adaptive education is a newer term for remedia...,https://library.educause.edu/resources/2017/1/...,NaN,NaN
1747,ADD,~,NaN,unconscious_bias,behavior,advanced_abbreviation,NaN,attention deficit hyperactivity disability (AD...,neurodivergent | ADHDer,person with ADHD | someone who has ADHD,...,NaN,NaN,NaN,(ADHD) | ADHDer,DCC,My ADD makes me really disorganized and forget...,Disorder is considered a medical term often re...,https://childmind.org/article/what-is-the-diff...,NaN,NaN
1750,attention deficit disorder,~s|~s|s,NaN,unconscious_bias,behavior,NaN,NaN,attention deficit disability ADD | attention d...,neurodivergent | ADHDer,person with ADD | someone who has ADHD,...,NaN,NaN,NaN,(ADD) | (ADHD),DCC,Attention Deficit Disorder is one of the most ...,Disorder is considered a medical term often re...,https://www.additudemag.com/3-types-of-adhd/,NaN,NaN
1751,autism spectrum disorder,~s|~s|s,NaN,unconscious_bias,behavior,NaN,NaN,being on the autism spectrum | having autism s...,Autistic | neurodivergent,person who has autism | someone with autism,...,NaN,NaN,NaN,NaN,DCC,They are enrolled in a program for potential e...,Disorder is considered a medical term often re...,https://jaapl.org/content/42/2/165,NaN,NaN
1770,neuroatypical,a,NaN,unconscious_bias,behavior,NaN,NaN,NaN,neurodivergent,person who is neurodivergent,...,NaN,NaN,NaN,NaN,DCC,When the brain's of neuroatypical people are s...,"While ""neuroatypical"" is not technically incor...",https://exceptionalindividuals.com/neurodivers...,NaN,NaN


In [ ]:
# Check Identity_first alternatives
df_rules[df_rules['Identity_first_pl'].notna()]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Identity_first,Alt_Sg_people_first,...,Alt_Pl_people_first,Alt_Pl_reframed,3rd_party_alternatives,False_Positives,3rd_party_source,Sample_Sentences,Notes,ToClarify,Generated Examples,Unnamed: 23
1494,birth-defect,s,NaN,unconscious_bias,advanced_ability,NaN,---name congenital health condition or disability,has a disability | congenital disability | has...,is disabled,NaN,...,NaN,NaN,NaN,NaN,DCC,"Birth defects can be diagnosed before, at, and...",Disability is preferred because of its more in...,https://www.lawinsider.com/dictionary/congenit...,NaN,NaN
1498,coexisting difficulty,~a|s,NaN,unconscious_bias,advanced_ability,NaN,NaN,co-occurring disability | concurrent disabilit...,disabled,NaN,...,NaN,NaN,NaN,NaN,DCC,Poor motor coordination and cognition are coex...,Disability is preferred because of its more in...,https://academic.oup.com/jdsde/article/19/2/18...,NaN,NaN
1782,unique personality,~a|~s,NaN,unconscious_bias,behavior,NaN,NaN,individuality | expressing their individuality,autonomous personality | autonomous thinker,NaN,...,NaN,NaN,NaN,NaN,DCC,Neurodivergent people exhibit unique personali...,"Depicting Disabled people as having ""unique"" p...",https://plato.stanford.edu/entries/cognitive-d...,NaN,NaN
1784,with ADHD,|s,NaN,unconscious_bias,behavior,NaN,NaN,NaN,ADHDer | neurodivergent ((person)),who has attention deficit hyperactivity disabi...,...,pople who have attention deficit hyperactivity...,NaN,NaN,NaN,DCC,Someone with ADHD may experience their ADHD di...,Disorder is considered a medical term often re...,https://www.additudemag.com/adhd-resilience-po...,NaN,NaN
1841,cognitive challenge,~a|~s,NaN,unconscious_bias,cognitive_perception,NaN,NaN,cognitive disability | cognition | cognitive s...,cognitively disabled | intellectually disabled,NaN,...,NaN,NaN,NaN,NaN,DCC,"After the equestrian fell off the horse, she w...",Disability is preferred because of its more in...,https://www.fcc.gov/cognitive-disabilities,NaN,NaN
1854,intellectual disorder,~a|~s,NaN,unconscious_bias,cognitive_perception,NaN,--- name the specific disability,intellectual disability,intellectually disabled,NaN,...,NaN,NaN,NaN,NaN,DCC,Our neighbor has an intellectual disorder and ...,Disorder is considered a medical term often re...,https://www.epa.gov/sites/default/files/2015-1...,NaN,NaN
1866,mild cognitive impairment,a|a|~s,NaN,unconscious_bias,cognitive_perception,NaN,--- describe the attributes of the disability,cognitive disability --- name the specific dis...,cognitively disabled | intellectually disabled,NaN,...,NaN,NaN,NaN,NaN,DCC,The patient with mild cognitive impairment is ...,Mild refers to functioning labels which the di...,https://www.michiganallianceforfamilies.org/co...,NaN,NaN
1969,person who is deaf,s||~v|~a,NaN,unconscious_bias,hearing,NaN,NaN,NaN,Deaf person,NaN,...,NaN,NaN,NaN,NaN,APA,NaN,NaN,NaN,NaN,NaN
1970,person with deafness,s||s,NaN,unconscious_bias,hearing,NaN,NaN,NaN,Deaf person,NaN,...,NaN,NaN,NaN,NaN,APA,NaN,NaN,NaN,NaN,NaN
1971,person with deafness and blindness,s||s||s,NaN,unconscious_bias,hearing,NaN,NaN,NaN,Deaf-Blind person,NaN,...,NaN,NaN,NaN,NaN,APA,NaN,NaN,NaN,NaN,NaN


### add (...) to
Refraimed columns:
Alt_Pl_people_first
Alt_Pl_reframed

Alt_Sg_people_first
Alt_Sg_reframed

In [ ]:
# add (...) for inspiration alternatives
df_rules["Alt_Pl_reframed"]=df_rules["Alt_Pl_reframed"].map(add_dots)

In [ ]:
# add (...) for inspiration alternatives
df_rules["Alt_Pl_people_first"]=df_rules["Alt_Pl_people_first"].map(add_dots)

In [ ]:
# add (...) for inspiration alternatives
df_rules["Alt_Sg_people_first"]=df_rules["Alt_Sg_people_first"].map(add_dots)

In [ ]:
# add (...) for inspiration alternatives
df_rules["Alt_Sg_reframed"]=df_rules["Alt_Sg_reframed"].map(add_dots)

In [ ]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2287 entries, 0 to 2286
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Lemma                   2287 non-null   object 
 1   Word_Type               2287 non-null   object 
 2   Priority                0 non-null      float64
 3   Category                2287 non-null   object 
 4   Primary_subcategory     2287 non-null   object 
 5   Secondary_subcategory   671 non-null    object 
 6   Alt_Field               775 non-null    object 
 7   Alt_Sg_Replacement      1296 non-null   object 
 8   Identity_first          56 non-null     object 
 9   Alt_Sg_people_first     199 non-null    object 
 10  Alt_Sg_reframed         322 non-null    object 
 11  Alt_Pl_collective_noun  58 non-null     object 
 12  Alt_Pl                  395 non-null    object 
 13  Identity_first_pl       25 non-null     object 
 14  Alt_Pl_people_first     129 non-null    

In [ ]:
test2= df_rules.query("Lemma=='go the extra mile'")

In [ ]:
test2

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives
1651,go the extra mile,~v||a|~s,unconscious_bias,agentic,competitive,"[think ahead, make a special effort, make cust...",,[],"[think ahead, make a special effort, make cust...",NaN


In [ ]:
pd.set_option('display.max_rows', None)

In [ ]:
test2["Alt_Sg_Replacement"]

227    they
Name: Alt_Sg_Replacement, dtype: object

In [ ]:
test2["Alt_Sg_reframed"]

227    None
Name: Alt_Sg_reframed, dtype: object

## Alternatives Single

### Single click+rephrase. Important order of columns with alternatives

In [ ]:
#order for single form of alternatives provided by Tracey
df_rules["Sg_all"]= df_rules.loc[:, ("Identity_first", "Alt_Sg_Replacement", "Alt_Field", "Alt_Pl_collective_noun", "Alt_Pl", "Alt_Sg_people_first", "Alt_Sg_reframed")].apply(
    lambda x: " | ".join(x.dropna().astype(str)), axis=1)

In [ ]:
# split alternatives in the row by " | " applying function split_form to convert string alternatives into array
df_rules["Sg_all_split"]=df_rules["Sg_all"].map(split_form)

In [ ]:
#check
df_rules.query("Lemma=='endurance'")

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Identity_first,Alt_Sg_people_first,...,3rd_party_alternatives,False_Positives,3rd_party_source,Sample_Sentences,Notes,ToClarify,Generated Examples,Unnamed: 23,Sg_all,Sg_all_split
1637,endurance,s,NaN,unconscious_bias,agentic,competitive,NaN,resilience | bearing | tolerance | forbearance...,NaN,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,resilience | bearing | tolerance | forbearance...,"[resilience, bearing, tolerance, forbearance, ..."


## Alternatives Plural

### Single click + rephrase. Important order of columns with alternatives

In [ ]:
#order for plural form of alternatives provided by Tracey
df_rules["Pl_all"]= df_rules.loc[:, ("Identity_first_pl", "Alt_Pl", "Alt_Pl_collective_noun", "Alt_Field", "Alt_Pl_people_first", "Alt_Pl_reframed")].apply(
    lambda x: " | ".join(x.dropna().astype(str)), axis=1)

In [ ]:
# split alternatives in the row by " | " applying function split_form
df_rules["Pl_all_split"]=df_rules["Pl_all"].map(split_form)

### Merge alternatives into one column

In [ ]:
source_col_loc = df_rules.columns.get_loc('Alt_Field') # column position starts from 0

In [ ]:
#merge alternatives in one column by sign  " | "
df_rules['alternatives']= df_rules.iloc[:,source_col_loc:source_col_loc+8].apply(
    lambda x: " | ".join(x.dropna().astype(str)), axis=1)

In [ ]:
# split alternatives in the row by " | " applying function split_form
df_rules["Alt_split"]=df_rules["alternatives"].map(split_form)

In [ ]:
df_rules.head()

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Identity_first,Alt_Sg_people_first,...,Notes,ToClarify,Generated Examples,Unnamed: 23,Sg_all,Sg_all_split,Pl_all,Pl_all_split,alternatives,Alt_split
0,big hair,~a|~s,NaN,ability-physicality,advanced_ability,NaN,---don't use for people. identify them by role...,NaN,NaN,None,...,NaN,NaN,NaN,NaN,---don't use for people. identify them by role...,[---don't use for people. identify them by rol...,---don't use for people. identify them by role...,[---don't use for people. identify them by rol...,---don't use for people. identify them by role...,[---don't use for people. identify them by rol...
1,brains before beauty,~s|conj|s,NaN,ability-physicality,advanced_ability,NaN,NaN,NaN,NaN,None,...,NaN,NaN,NaN,NaN,,[],,[],,[]
2,chubby,a,NaN,ability-physicality,advanced_ability,NaN,---don't use for people. identify them by role...,NaN,NaN,None,...,NaN,NaN,NaN,NaN,---don't use for people. identify them by role...,[---don't use for people. identify them by rol...,---don't use for people. identify them by role...,[---don't use for people. identify them by rol...,---don't use for people. identify them by role...,[---don't use for people. identify them by rol...
3,fat,a,NaN,ability-physicality,advanced_ability,NaN,---don't use for people. identify them by role...,NaN,NaN,None,...,NaN,NaN,NaN,NaN,---don't use for people. identify them by role...,[---don't use for people. identify them by rol...,---don't use for people. identify them by role...,[---don't use for people. identify them by rol...,---don't use for people. identify them by role...,[---don't use for people. identify them by rol...
4,good-looking,~a,NaN,ability-physicality,advanced_ability,NaN,---don't use for people. identify them by role...,NaN,NaN,None,...,NaN,NaN,NaN,NaN,---don't use for people. identify them by role...,[---don't use for people. identify them by rol...,---don't use for people. identify them by role...,[---don't use for people. identify them by rol...,---don't use for people. identify them by role...,[---don't use for people. identify them by rol...


### Cleaning of alternatives

In [ ]:
# strip alternatives
df_rules["Alt_split"]=df_rules["Alt_split"].map(clean_str)

In [ ]:
# strip alternatives
df_rules["Sg_all_split"]=df_rules["Sg_all_split"].map(clean_str)

In [ ]:
# strip alternatives
df_rules["Pl_all_split"]=df_rules["Pl_all_split"].map(clean_str)

### Create subsets of dataframes

In [ ]:
df_rules = df_rules[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Secondary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split", "False_Positives"]]

In [ ]:
df_rules[df_rules["Word_Type"]=="=v"]

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives


In [ ]:
df_rules[df_rules["Word_Type"]=="-v"]

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives


In [ ]:
df_verbs = df_rules[(df_rules["Word_Type"]=="v") | (df_rules["Word_Type"]=="=v")]

In [ ]:
df_verbs.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 93 entries, 20 to 2256
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Lemma                  93 non-null     object
 1   Word_Type              93 non-null     object
 2   Category               93 non-null     object
 3   Primary_subcategory    93 non-null     object
 4   Secondary_subcategory  64 non-null     object
 5   Sg_all_split           93 non-null     object
 6   Pl_all                 93 non-null     object
 7   Pl_all_split           93 non-null     object
 8   Alt_split              93 non-null     object
 9   False_Positives        8 non-null      object
dtypes: object(10)
memory usage: 8.0+ KB


In [ ]:
df_verbs["Category"].unique()

array(['gendered', 'inclusive', 'style', 'unconscious_bias'], dtype=object)

In [ ]:
df_verbs[df_verbs["Category"]=='style']

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives
1311,leverage,v,style,advanced_hollow,NaN,"[use, capitalize on]",,[],"[use, capitalize on]",NaN
1329,synergize,v,style,advanced_hollow,NaN,[work together],,[],[work together],NaN
1336,utilize,v,style,advanced_hollow,NaN,[use],,[],[use],NaN
1403,surpass,v,style,exaggerating,NaN,"[refine, improve upon, transcend, do better th...",,[],"[refine, improve upon, transcend, do better th...",surpass expectations | surpassed expectations
1426,believe me,v,style,filler,NaN,[-],-,[-],[-],NaN
1467,seem,v,style,filler,NaN,[-],-,[-],[-],NaN


### Check duplicates&homonyms

In [ ]:
df_rules[df_rules["Lemma"].duplicated(keep = False)]

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives
20,man,v,gendered,advanced_hidden_image,NaN,"[staff, crew, occupy, work, use]",,[],"[staff, crew, occupy, work, use]",NaN
91,hysteric,a,gendered,female_stereotype,NaN,"[very upset, angry, agitated]",,[],"[very upset, angry, agitated]",NaN
92,hysteric,~s,gendered,female_stereotype,NaN,"[irrational behavior, overwhelming fear, expre...",,[],"[irrational behavior, overwhelming fear, expre...",NaN
97,nag,v,gendered,female_stereotype,NaN,"[criticize, annoy, complain, repeatedly commen...",,[],"[criticize, annoy, complain, repeatedly commen...",NaN
98,nag,s,gendered,female_stereotype,NaN,"[critic, they, critics, person who likes shari...",critics | people who like sharing their reserv...,"[critics, people who like sharing their reserv...","[critic, they, person who likes sharing their ...",NaN
145,pioneer,s,gendered,function,NaN,"[inspiring personality, visionary, inspiring p...",inspiring personalities | visionaries | pionee...,"[inspiring personalities, visionaries, pioneer...","[inspiring personality, visionary, person who ...",NaN
168,lead,s,gendered,leadership,generic_plural,"[guide, mentor, facilitator, supporter, leads,...",leads | mentors | facilitators | supporters | ...,"[leads, mentors, facilitators, supporters, peo...","[guide, mentor, facilitator, supporter, person...",lead generation | generate leads | generate a ...
169,lead,v,gendered,leadership,rough,"[provide direction to, counsel, mentor, guide,...",,[],"[provide direction to, counsel, mentor, guide,...",lead a (...) life | lead (...) life | lead now...
327,impact,s,inclusive,communal,cultural,[],,[],[],NaN
418,impact,v,inclusive,d_and_i,NaN,[],,[],[],NaN


### Homonyms

In [ ]:
# check how many homonyms
df_rules["Lemma"].duplicated(keep = False).sum()

64

In [ ]:
df_homonyms = df_rules[df_rules["Lemma"].duplicated(keep = False)]

In [ ]:
#check first 5 rows
df_homonyms

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives
20,man,v,gendered,advanced_hidden_image,NaN,"[staff, crew, occupy, work, use]",,[],"[staff, crew, occupy, work, use]",NaN
91,hysteric,a,gendered,female_stereotype,NaN,"[very upset, angry, agitated]",,[],"[very upset, angry, agitated]",NaN
92,hysteric,~s,gendered,female_stereotype,NaN,"[irrational behavior, overwhelming fear, expre...",,[],"[irrational behavior, overwhelming fear, expre...",NaN
97,nag,v,gendered,female_stereotype,NaN,"[criticize, annoy, complain, repeatedly commen...",,[],"[criticize, annoy, complain, repeatedly commen...",NaN
98,nag,s,gendered,female_stereotype,NaN,"[critic, they, critics, person who likes shari...",critics | people who like sharing their reserv...,"[critics, people who like sharing their reserv...","[critic, they, person who likes sharing their ...",NaN
145,pioneer,s,gendered,function,NaN,"[inspiring personality, visionary, inspiring p...",inspiring personalities | visionaries | pionee...,"[inspiring personalities, visionaries, pioneer...","[inspiring personality, visionary, person who ...",NaN
168,lead,s,gendered,leadership,generic_plural,"[guide, mentor, facilitator, supporter, leads,...",leads | mentors | facilitators | supporters | ...,"[leads, mentors, facilitators, supporters, peo...","[guide, mentor, facilitator, supporter, person...",lead generation | generate leads | generate a ...
169,lead,v,gendered,leadership,rough,"[provide direction to, counsel, mentor, guide,...",,[],"[provide direction to, counsel, mentor, guide,...",lead a (...) life | lead (...) life | lead now...
327,impact,s,inclusive,communal,cultural,[],,[],[],NaN
418,impact,v,inclusive,d_and_i,NaN,[],,[],[],NaN


In [ ]:
#create subset of dataframe to drop unnecessary columns
df_homonyms = df_homonyms[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]

In [ ]:
df_homonyms.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 64 entries, 20 to 2168
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                64 non-null     object
 1   Word_Type            64 non-null     object
 2   Category             64 non-null     object
 3   Primary_subcategory  64 non-null     object
 4   Alt_split            64 non-null     object
dtypes: object(5)
memory usage: 3.0+ KB


In [ ]:
#save homonyms to csv file after preprocessing
df_homonyms.to_csv("homonyms_words.csv", index=False)

In [ ]:
replace_tolist(r"homonyms_words.csv")

'homonyms_words.csv'

### Remove homonyms from dataframe for the next steps pre-processing

In [ ]:
homonyms_list = list(df_rules[df_rules["Lemma"].duplicated()]["Lemma"])

In [ ]:
homonyms_list

['hysteric',
 'nag',
 'lead',
 'impact',
 'must-have',
 'ninja',
 'dummy',
 'handicap',
 'elderly',
 'challenge',
 'direct',
 'drive',
 'fight',
 'force',
 'objective',
 'pioneer',
 'wacko',
 'stupid',
 'psycho',
 'schizo',
 'sass',
 'female',
 'male',
 'man',
 'queer',
 'dwarf',
 'cripple',
 'Aryan',
 'Eskimo',
 'grandfather',
 'indian',
 'master']

In [ ]:
df_rules = df_rules.query("Lemma != @homonyms_list")

### Binary_pronouns

In [ ]:
df_binary_pronouns = df_rules[df_rules["Primary_subcategory"]=="binary_pronouns"]

In [ ]:
df_binary_pronouns.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 27 entries, 51 to 77
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Lemma                  27 non-null     object
 1   Word_Type              27 non-null     object
 2   Category               27 non-null     object
 3   Primary_subcategory    27 non-null     object
 4   Secondary_subcategory  0 non-null      object
 5   Sg_all_split           27 non-null     object
 6   Pl_all                 27 non-null     object
 7   Pl_all_split           27 non-null     object
 8   Alt_split              27 non-null     object
 9   False_Positives        0 non-null      object
dtypes: object(10)
memory usage: 2.3+ KB


In [ ]:
df_binary_pronouns.head()

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives
51,he,~s,gendered,binary_pronouns,NaN,"[they, ((given name)), ((given name family nam...",,[],"[they, ((given name)), ((given name family nam...",NaN
52,he or she,~s||~s,gendered,binary_pronouns,NaN,"[they, ((given name)), ((given name family nam...",,[],"[they, ((given name)), ((given name family nam...",NaN
53,he/she,~s||~s,gendered,binary_pronouns,NaN,"[they, ((given name)), ((given name family nam...",,[],"[they, ((given name)), ((given name family nam...",NaN
54,her,~s,gendered,binary_pronouns,NaN,"[them, their, ((given name)), ((given name fam...",,[],"[them, their, ((given name)), ((given name fam...",NaN
55,her or him,~s||~s,gendered,binary_pronouns,NaN,"[them, ((given name)), ((given name family nam...",,[],"[them, ((given name)), ((given name family nam...",NaN


In [ ]:
# separate one term binary pronouns
df_binary_pronouns_word =separate_binary_pronouns(df_binary_pronouns)

In [ ]:
#check the results
df_binary_pronouns_word

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
0,he,~s,gendered,binary_pronouns,"[they, ((given name)), ((given name family nam...",,[],"[they, ((given name)), ((given name family nam..."
1,her,~s,gendered,binary_pronouns,"[them, their, ((given name)), ((given name fam...",,[],"[them, their, ((given name)), ((given name fam..."
2,hers,~s,gendered,binary_pronouns,"[theirs, ((given name's))]",,[],"[theirs, ((given name's))]"
3,herself,~s,gendered,binary_pronouns,"[themself, theirself, self]",,[],"[themself, theirself, self]"
4,him,~s,gendered,binary_pronouns,"[them, ((given name)), ((given name family nam...",,[],"[them, ((given name)), ((given name family nam..."
5,himself,~s,gendered,binary_pronouns,"[themself, theirself, self]",,[],"[themself, theirself, self]"
6,his,~s,gendered,binary_pronouns,"[their, ((given name's)), theirs]",,[],"[their, ((given name's)), theirs]"
7,she,~s,gendered,binary_pronouns,"[they, ((given name)), ((given name family nam...",,[],"[they, ((given name)), ((given name family nam..."


In [ ]:
#create subset of dataframe to drop unnecessary columns
df_binary_pronouns_word = df_binary_pronouns_word[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Sg_all_split", "Pl_all_split", "Alt_split"]]

In [ ]:
#save one term binary_pronouns to csv file after preprocessing
df_binary_pronouns_word.to_csv("ub_singular_they.csv", index=False)

In [ ]:
df_binary_pronouns_word.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                8 non-null      object
 1   Word_Type            8 non-null      object
 2   Category             8 non-null      object
 3   Primary_subcategory  8 non-null      object
 4   Sg_all_split         8 non-null      object
 5   Pl_all_split         8 non-null      object
 6   Alt_split            8 non-null      object
dtypes: object(7)
memory usage: 576.0+ bytes


In [ ]:
replace_tolist(r"ub_singular_they.csv")

'ub_singular_they.csv'

In [ ]:
#test read of dataframe
df_test = pd.read_csv("ub_singular_they.csv")

In [ ]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                8 non-null      object
 1   Word_Type            8 non-null      object
 2   Category             8 non-null      object
 3   Primary_subcategory  8 non-null      object
 4   Sg_all_split         8 non-null      object
 5   Pl_all_split         8 non-null      object
 6   Alt_split            8 non-null      object
dtypes: object(7)
memory usage: 576.0+ bytes


In [ ]:
df_test

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all_split,Alt_split
0,he,~s,gendered,binary_pronouns,"['they', '((given name))', '((given name famil...",[],"['they', '((given name))', '((given name famil..."
1,her,~s,gendered,binary_pronouns,"['them', 'their', '((given name))', '((given n...",[],"['them', 'their', '((given name))', '((given n..."
2,hers,~s,gendered,binary_pronouns,"['theirs', ""((given name's))""]",[],"['theirs', ""((given name's))""]"
3,herself,~s,gendered,binary_pronouns,"['themself', 'theirself', 'self']",[],"['themself', 'theirself', 'self']"
4,him,~s,gendered,binary_pronouns,"['them', '((given name))', '((given name famil...",[],"['them', '((given name))', '((given name famil..."
5,himself,~s,gendered,binary_pronouns,"['themself', 'theirself', 'self']",[],"['themself', 'theirself', 'self']"
6,his,~s,gendered,binary_pronouns,"['their', ""((given name's))"", 'theirs']",[],"['their', ""((given name's))"", 'theirs']"
7,she,~s,gendered,binary_pronouns,"['they', '((given name))', '((given name famil...",[],"['they', '((given name))', '((given name famil..."


### Remove one term binary_pronouns from dataframe for the next steps pre-processing

In [ ]:
#remove binary pronouns
binary_list = list(df_binary_pronouns_word["Lemma"])

In [ ]:
len(binary_list)

8

In [ ]:
df_rules = df_rules.query("Lemma != @binary_list")

In [ ]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2215 entries, 0 to 2286
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Lemma                  2215 non-null   object
 1   Word_Type              2215 non-null   object
 2   Category               2215 non-null   object
 3   Primary_subcategory    2215 non-null   object
 4   Secondary_subcategory  654 non-null    object
 5   Sg_all_split           2215 non-null   object
 6   Pl_all                 2215 non-null   object
 7   Pl_all_split           2215 non-null   object
 8   Alt_split              2215 non-null   object
 9   False_Positives        370 non-null    object
dtypes: object(10)
memory usage: 190.4+ KB


### Acronyms (abbreviations)

In [ ]:
#check word_type
df_rules["Word_Type"].unique()

array(['~a|~s', '~s|conj|s', 'a', '~a', '~s', '~s|v|s', 'adv|v|||s|v|v|a',
       's', '|~s||', '~s|~s', 's|', 'adv', '~s|', '~s|s', '|s|~v||s',
       '~s||~s', 's||s', 's|s', 'v||~s', '|~s', 'a|~s', 'v|s', '~s|||s',
       'v', '=|', 'a|s', 'v||~s|', '~s||a|~s', '~s|||~s', 'v|~s',
       '|~s|||~s', 's|s|s', 'v||s', '||~s', '||a|~s', 'v|a', '|s',
       's||~s', '~a|s', 'a|||s', 'a|||~s', 's||', '|', '~', 's|||~s',
       'a|~s|~s|s', 's||s|', 'a||s', '=', '~adv', 'a|', '||s|||s',
       'v|||a|~s', 'v|~s|||~s', '~a|||', '|a|~s', 'v|a||~s', '~v|',
       'a|~a', '~s|v|~s|v', '~||v|||~a', 'v|adv||s|v|a||~s', 'v||v||~s|',
       '|v||~s', '||a|s', 'v|~s|a', '~s||s', 's||~', '|a', '~a|~s|s',
       'v|||s', '||s', 'conj', '||', '~v|~v|~v|', 'v|', '||~s||~v|a',
       '~s|~v', 's|v', '||~a|~s', '|s|~v', '||~v', '~s||', 's|v|v|a|',
       '|s|v|', '||v||v|v|', '~|~||~s|~v|', '~|~|', 'a||s|||~s',
       'v|~a|~s', 'a|~s|~s', 'adv|a|s', 'v||~s|s', 'v|~s|~s', '~a|s|~s',
       '~a|~v', '~a|'

In [ ]:
#df_abbr = df_rules[(df_rules["Word_Type"]=='abbr') | (df_rules["Word_Type"]=='acr') | (df_rules["Word_Type"]=='-acr') | (df_rules["Word_Type"]=='-abbr')]

In [ ]:
# create a dataframe with abbreviations
df_abbr = df_rules[(df_rules["Primary_subcategory"]=='advanced_abbreviation') | (df_rules["Primary_subcategory"]=='abbreviation')]

In [ ]:
df_abbr.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 306 entries, 972 to 1277
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Lemma                  306 non-null    object
 1   Word_Type              306 non-null    object
 2   Category               306 non-null    object
 3   Primary_subcategory    306 non-null    object
 4   Secondary_subcategory  297 non-null    object
 5   Sg_all_split           306 non-null    object
 6   Pl_all                 306 non-null    object
 7   Pl_all_split           306 non-null    object
 8   Alt_split              306 non-null    object
 9   False_Positives        303 non-null    object
dtypes: object(10)
memory usage: 26.3+ KB


In [ ]:
#create subset of dataframe to drop unnecessary columns
df_abbr = df_abbr[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]

In [ ]:
#save acronyms (abbreviations) to csv file after preprocessing
df_abbr.to_csv("abbreviations.csv", index=False)

In [ ]:
replace_tolist(r"abbreviations.csv")

'abbreviations.csv'

### Remove acronymes (abbr) from dataframe for the next steps pre-processing

In [ ]:
#remove abbr
abbr_list = list(df_abbr["Lemma"])

In [ ]:
df_rules = df_rules.query("Lemma != @abbr_list")

In [ ]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1909 entries, 0 to 2286
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Lemma                  1909 non-null   object
 1   Word_Type              1909 non-null   object
 2   Category               1909 non-null   object
 3   Primary_subcategory    1909 non-null   object
 4   Secondary_subcategory  357 non-null    object
 5   Sg_all_split           1909 non-null   object
 6   Pl_all                 1909 non-null   object
 7   Pl_all_split           1909 non-null   object
 8   Alt_split              1909 non-null   object
 9   False_Positives        67 non-null     object
dtypes: object(10)
memory usage: 164.1+ KB


## Openly_discriminating

In [ ]:
df_openly_discrim = df_rules[df_rules["Category"]=="openly_discriminating"]

In [ ]:
#info open_discrim
df_openly_discrim.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 451 entries, 520 to 971
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Lemma                  451 non-null    object
 1   Word_Type              451 non-null    object
 2   Category               451 non-null    object
 3   Primary_subcategory    451 non-null    object
 4   Secondary_subcategory  1 non-null      object
 5   Sg_all_split           451 non-null    object
 6   Pl_all                 451 non-null    object
 7   Pl_all_split           451 non-null    object
 8   Alt_split              451 non-null    object
 9   False_Positives        1 non-null      object
dtypes: object(10)
memory usage: 38.8+ KB


In [ ]:
df_openly_discrim.head()

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives
520,freak,s,openly_discriminating,ableism,NaN,[-],-,[-],[-],NaN
521,fucktard,s,openly_discriminating,ableism,NaN,[-],-,[-],[-],NaN
522,herp-derp,s,openly_discriminating,ableism,NaN,[-],-,[-],[-],NaN
523,libtard,s,openly_discriminating,ableism,NaN,[-],-,[-],[-],NaN
524,supercrip,s,openly_discriminating,ableism,NaN,[-],-,[-],[-],NaN


In [ ]:
#create subset of dataframe to drop unnecessary columns
df_open_dis_word = df_openly_discrim[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]
#df_open_dis_sentence = df_open_dis_sentence[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]

In [ ]:
df_open_dis_word.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 451 entries, 520 to 971
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                451 non-null    object
 1   Word_Type            451 non-null    object
 2   Category             451 non-null    object
 3   Primary_subcategory  451 non-null    object
 4   Alt_split            451 non-null    object
dtypes: object(5)
memory usage: 21.1+ KB


In [ ]:
df_open_dis_word.head()

,Lemma,Word_Type,Category,Primary_subcategory,Alt_split
520,freak,s,openly_discriminating,ableism,[-]
521,fucktard,s,openly_discriminating,ableism,[-]
522,herp-derp,s,openly_discriminating,ableism,[-]
523,libtard,s,openly_discriminating,ableism,[-]
524,supercrip,s,openly_discriminating,ableism,[-]


In [ ]:
#save to csv file after preprocessing
df_open_dis_word.to_csv("open_dis_words.csv", index=False)

In [ ]:
replace_tolist(r"open_dis_words.csv")

'open_dis_words.csv'

In [ ]:
#replace_tolist(r"open_dis_sentences.csv")

## Style

In [ ]:
df_style = df_rules[df_rules["Category"]=="style"]

In [ ]:
#info style
df_style.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 205 entries, 1278 to 1486
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Lemma                  205 non-null    object
 1   Word_Type              205 non-null    object
 2   Category               205 non-null    object
 3   Primary_subcategory    205 non-null    object
 4   Secondary_subcategory  8 non-null      object
 5   Sg_all_split           205 non-null    object
 6   Pl_all                 205 non-null    object
 7   Pl_all_split           205 non-null    object
 8   Alt_split              205 non-null    object
 9   False_Positives        22 non-null     object
dtypes: object(10)
memory usage: 17.6+ KB


In [ ]:
#create subset of dataframe to drop unnecessary columns
df_style_word = df_style[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Sg_all_split", "Pl_all_split", "Alt_split"]]
#df_style_sentence = df_style_sentence[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Sg_all_split", "Pl_all_split", "Alt_split"]]

In [ ]:
df_style_word.query("Word_Type == '=s'")

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all_split,Alt_split


In [ ]:
df_style_word.query("Word_Type == '-s'")

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all_split,Alt_split


In [ ]:
df_style_word.query("Word_Type == '~s'")

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all_split,Alt_split


In [ ]:
df_style_word["Word_Type"].unique()

array(['~a|~s', '~adv', 'a|~s', 's', 'a|', '||s|||s', 'v|||a|~s',
       'v|~s|||~s', '~s|s', 'a|s', 'v||s', 'a', '~a|||', '~a|s', '|a|~s',
       'v|a||~s', '~v|', 'a|~a', '~s|v|~s|v', '~||v|||~a', '~a', 'v',
       'v|adv||s|v|a||~s', 'v||~s|', 'v||v||~s|', '|v||~s', '||a|s',
       'v||~s', 'v|~s|a', '~s||s', 's||~', '|a', 's|', '~a|~s|s', 'v|||s',
       'adv', '||s', '|s', '|~s', 'conj', '||', '||~s', '~v|~v|~v|', 'v|',
       '~', '||~s||~v|a', '~s|~v', 's|v', '||~a|~s', '|', '|s|~v', '||~v',
       '~s||', 's|v|v|a|', '|s|v|', '||v||v|v|', '~|~||~s|~v|', '~|~|'],
      dtype=object)

In [ ]:
df_style_word_noun = df_style_word.query("Word_Type == 's'")

In [ ]:
df_style_word_noun.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 30 entries, 1281 to 1411
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                30 non-null     object
 1   Word_Type            30 non-null     object
 2   Category             30 non-null     object
 3   Primary_subcategory  30 non-null     object
 4   Sg_all_split         30 non-null     object
 5   Pl_all_split         30 non-null     object
 6   Alt_split            30 non-null     object
dtypes: object(7)
memory usage: 1.9+ KB


In [ ]:
df_style_word_no_noun = df_style_word.query("Word_Type != 's'")

In [ ]:
df_style_word_no_noun.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 175 entries, 1278 to 1486
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                175 non-null    object
 1   Word_Type            175 non-null    object
 2   Category             175 non-null    object
 3   Primary_subcategory  175 non-null    object
 4   Sg_all_split         175 non-null    object
 5   Pl_all_split         175 non-null    object
 6   Alt_split            175 non-null    object
dtypes: object(7)
memory usage: 10.9+ KB


In [ ]:
#save to csv file after preprocessing
df_style_word_noun.to_csv("style_noun_words.csv", index=False)

In [ ]:
#save to csv file after preprocessing
df_style_word_no_noun.to_csv("style_no_noun_words.csv", index=False)

In [ ]:
replace_tolist(r"style_noun_words.csv")

'style_noun_words.csv'

In [ ]:
replace_tolist(r"style_no_noun_words.csv")

'style_no_noun_words.csv'

## Inclusive

In [ ]:
df_inclusive = df_rules[df_rules["Category"]=="inclusive"]

In [ ]:
#info inclusive
df_inclusive.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 247 entries, 271 to 519
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Lemma                  247 non-null    object
 1   Word_Type              247 non-null    object
 2   Category               247 non-null    object
 3   Primary_subcategory    247 non-null    object
 4   Secondary_subcategory  116 non-null    object
 5   Sg_all_split           247 non-null    object
 6   Pl_all                 247 non-null    object
 7   Pl_all_split           247 non-null    object
 8   Alt_split              247 non-null    object
 9   False_Positives        2 non-null      object
dtypes: object(10)
memory usage: 21.2+ KB


In [ ]:
df_inclusive.head()

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives
271,affectionate,a,inclusive,communal,caring,[],,[],[],NaN
272,agree,v,inclusive,communal,cooperative,[],,[],[],NaN
273,agreeable,a,inclusive,communal,cooperative,[],,[],[],NaN
274,care,s,inclusive,communal,caring,[],,[],[],NaN
275,cheer,v,inclusive,communal,cooperative,[],,[],[],NaN


In [ ]:
#create subset of dataframe to drop unnecessary columns
df_inclusive_word = df_inclusive[['Lemma', "Word_Type", "Category", "Primary_subcategory"]]
#df_inclusive_sentence = df_inclusive_sentence[['Lemma', "Word_Type", "Category", "Primary_subcategory"]]

In [ ]:
#save to csv file after preprocessing
df_inclusive_word.to_csv("inclusive_words.csv", index=False)

In [ ]:
replace_tolist(r"inclusive_words.csv")

'inclusive_words.csv'

## Gendered

In [ ]:
df_gendered = df_rules[df_rules["Category"]=="gendered"]

In [ ]:
#info gendered
df_gendered.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 241 entries, 14 to 270
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Lemma                  241 non-null    object
 1   Word_Type              241 non-null    object
 2   Category               241 non-null    object
 3   Primary_subcategory    241 non-null    object
 4   Secondary_subcategory  58 non-null     object
 5   Sg_all_split           241 non-null    object
 6   Pl_all                 241 non-null    object
 7   Pl_all_split           241 non-null    object
 8   Alt_split              241 non-null    object
 9   False_Positives        9 non-null      object
dtypes: object(10)
memory usage: 20.7+ KB


### Check if subcategory name is missing and set the parent category name to subcategory if the subategory name is missing¶

In [ ]:
#check missing values in Secondary subcategory
df_gendered[df_gendered['Secondary_subcategory'].isna()]

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives
14,boldly go where no man have go before,adv|v|||s|v|v|a,gendered,advanced_hidden_image,NaN,[boldly go where no one has gone before],,[],[boldly go where no one has gone before],NaN
15,brotherhood,s,gendered,advanced_hidden_image,NaN,"[kinship, community, religious order, associat...",kinship | communities | religious orders | ass...,"[kinship, communities, religious orders, assoc...","[kinship, community, religious order, associat...",NaN
16,common man,~a|~s,gendered,advanced_hidden_image,NaN,"[average person, average people, common person...",,[],"[average person, average people, common person...",NaN
17,every man for himself,|~s||,gendered,advanced_hidden_image,NaN,"[everyone for themself, everyone for themselves]",everyone for themselves,[everyone for themselves],"[everyone for themself, everyone for themselves]",NaN
18,fatherland,s,gendered,advanced_hidden_image,NaN,"[native land, homeland]",,[],"[native land, homeland]",NaN
19,guerilla marketing,~s|~s,gendered,advanced_hidden_image,NaN,"[low-cost, non-traditional advertising strateg...",,[],"[low-cost, non-traditional advertising strateg...",NaN
21,man up,s|,gendered,advanced_hidden_image,NaN,"[deal with it, toughen up, be brave, pull up y...",,[],"[deal with it, toughen up, be brave, pull up y...",NaN
22,man-day,s,gendered,advanced_hidden_image,NaN,"[person-day, person-days]",person-days,[person-days],"[person-day, person-days]",NaN
23,man-hour,s,gendered,advanced_hidden_image,NaN,"[person-hour, labor hour, hour of effort, pers...",person-hours | labor hours | hours of effort |...,"[person-hours, labor hours, hours of effort, a...","[person-hour, labor hour, hour of effort, amou...",NaN
24,man-hunt,s,gendered,advanced_hidden_image,NaN,"[intensive search for a person, organized sear...",intensive searches for a people | organized se...,"[intensive searches for a people, organized se...","[intensive search for a person, organized sear...",NaN


In [ ]:
#fill the missing values in "Secondary subcategory" by values from "Primary_Category"
df_gendered.Secondary_subcategory.fillna(df_gendered.Primary_subcategory, inplace=True)

<ipython-input-151-d75efa2208ee>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_gendered.Secondary_subcategory.fillna(df_gendered.Primary_subcategory, inplace=True)


In [ ]:
#check missing values in Secondary subcategory
df_gendered[df_gendered['Secondary_subcategory'].isna()]

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives


In [ ]:
df_gendered.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 241 entries, 14 to 270
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Lemma                  241 non-null    object
 1   Word_Type              241 non-null    object
 2   Category               241 non-null    object
 3   Primary_subcategory    241 non-null    object
 4   Secondary_subcategory  241 non-null    object
 5   Sg_all_split           241 non-null    object
 6   Pl_all                 241 non-null    object
 7   Pl_all_split           241 non-null    object
 8   Alt_split              241 non-null    object
 9   False_Positives        9 non-null      object
dtypes: object(10)
memory usage: 20.7+ KB


In [ ]:
df_gendered[df_gendered["Secondary_subcategory"].notnull()]

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives
14,boldly go where no man have go before,adv|v|||s|v|v|a,gendered,advanced_hidden_image,advanced_hidden_image,[boldly go where no one has gone before],,[],[boldly go where no one has gone before],NaN
15,brotherhood,s,gendered,advanced_hidden_image,advanced_hidden_image,"[kinship, community, religious order, associat...",kinship | communities | religious orders | ass...,"[kinship, communities, religious orders, assoc...","[kinship, community, religious order, associat...",NaN
16,common man,~a|~s,gendered,advanced_hidden_image,advanced_hidden_image,"[average person, average people, common person...",,[],"[average person, average people, common person...",NaN
17,every man for himself,|~s||,gendered,advanced_hidden_image,advanced_hidden_image,"[everyone for themself, everyone for themselves]",everyone for themselves,[everyone for themselves],"[everyone for themself, everyone for themselves]",NaN
18,fatherland,s,gendered,advanced_hidden_image,advanced_hidden_image,"[native land, homeland]",,[],"[native land, homeland]",NaN
19,guerilla marketing,~s|~s,gendered,advanced_hidden_image,advanced_hidden_image,"[low-cost, non-traditional advertising strateg...",,[],"[low-cost, non-traditional advertising strateg...",NaN
21,man up,s|,gendered,advanced_hidden_image,advanced_hidden_image,"[deal with it, toughen up, be brave, pull up y...",,[],"[deal with it, toughen up, be brave, pull up y...",NaN
22,man-day,s,gendered,advanced_hidden_image,advanced_hidden_image,"[person-day, person-days]",person-days,[person-days],"[person-day, person-days]",NaN
23,man-hour,s,gendered,advanced_hidden_image,advanced_hidden_image,"[person-hour, labor hour, hour of effort, pers...",person-hours | labor hours | hours of effort |...,"[person-hours, labor hours, hours of effort, a...","[person-hour, labor hour, hour of effort, amou...",NaN
24,man-hunt,s,gendered,advanced_hidden_image,advanced_hidden_image,"[intensive search for a person, organized sear...",intensive searches for a people | organized se...,"[intensive searches for a people, organized se...","[intensive search for a person, organized sear...",NaN


In [ ]:
df_gendered["Secondary_subcategory"].unique()

array(['advanced_hidden_image', 'binary_pronouns', 'female_stereotype',
       'function', 'generic_plural', 'leadership', 'male_stereotype',
       'titles'], dtype=object)

In [ ]:
df_gendered["Primary_subcategory"].unique()

array(['advanced_hidden_image', 'binary_pronouns', 'female_stereotype',
       'function', 'leadership', 'male_stereotype', 'titles'],
      dtype=object)

In [ ]:
df_gendered.query("Word_Type == '=s'")

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives


In [ ]:
df_gendered.query("Word_Type == '-s'")

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives


In [ ]:
df_gendered.query("Word_Type == '~s'")

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives
86,histrionics,~s,gendered,female_stereotype,female_stereotype,"[histrionic personality disorder, person with ...",histrionic personality disorder,[histrionic personality disorder],"[histrionic personality disorder, person with ...",histrionic personality disorder
87,housekeeping,~s,gendered,female_stereotype,female_stereotype,"[maintenance, cleanup, household management, d...",,[],"[maintenance, cleanup, household management, d...",NaN
94,hysterics,~s,gendered,female_stereotype,female_stereotype,"[frustration, panic, excitement, disappointmen...",,[],"[frustration, panic, excitement, disappointmen...",NaN


In [ ]:
df_gendered_word_noun = df_gendered.query("Word_Type == 's' or Word_Type == '~s'")

In [ ]:
df_gendered_word_noun.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 148 entries, 15 to 270
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Lemma                  148 non-null    object
 1   Word_Type              148 non-null    object
 2   Category               148 non-null    object
 3   Primary_subcategory    148 non-null    object
 4   Secondary_subcategory  148 non-null    object
 5   Sg_all_split           148 non-null    object
 6   Pl_all                 148 non-null    object
 7   Pl_all_split           148 non-null    object
 8   Alt_split              148 non-null    object
 9   False_Positives        8 non-null      object
dtypes: object(10)
memory usage: 12.7+ KB


In [ ]:
df_gendered_word_noun.query("Lemma=='engineer'")

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives
227,engineer,s,gendered,titles,generic_plural,"[they, engineering, development, engineering t...",engineers | engineering team | development tea...,"[engineers, engineering team, development team...","[engineering, development, they, person fillin...",NaN


In [ ]:
df_gendered_word_no_noun = df_gendered.query("Word_Type != 's'")

In [ ]:
df_gendered_word_no_noun.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 96 entries, 14 to 267
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Lemma                  96 non-null     object
 1   Word_Type              96 non-null     object
 2   Category               96 non-null     object
 3   Primary_subcategory    96 non-null     object
 4   Secondary_subcategory  96 non-null     object
 5   Sg_all_split           96 non-null     object
 6   Pl_all                 96 non-null     object
 7   Pl_all_split           96 non-null     object
 8   Alt_split              96 non-null     object
 9   False_Positives        2 non-null      object
dtypes: object(10)
memory usage: 8.2+ KB


In [ ]:
df_gendered_word_no_noun = df_gendered_word_no_noun.query("Word_Type != '~s'")

In [ ]:
df_gendered_word_no_noun.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 93 entries, 14 to 267
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Lemma                  93 non-null     object
 1   Word_Type              93 non-null     object
 2   Category               93 non-null     object
 3   Primary_subcategory    93 non-null     object
 4   Secondary_subcategory  93 non-null     object
 5   Sg_all_split           93 non-null     object
 6   Pl_all                 93 non-null     object
 7   Pl_all_split           93 non-null     object
 8   Alt_split              93 non-null     object
 9   False_Positives        1 non-null      object
dtypes: object(10)
memory usage: 8.0+ KB


In [ ]:
#drop unnecessary column "Pl_all"
df_gendered_word_noun = df_gendered_word_noun.drop(labels=["Pl_all"], axis=1)
df_gendered_word_no_noun = df_gendered_word_no_noun.drop(labels=["Pl_all"], axis=1)


In [ ]:
df_gendered_word_noun.query("Lemma=='engineer'")

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all_split,Alt_split,False_Positives
227,engineer,s,gendered,titles,generic_plural,"[they, engineering, development, engineering t...","[engineers, engineering team, development team...","[engineering, development, they, person fillin...",NaN


In [ ]:
#save to csv file after preprocessing
df_gendered_word_noun.to_csv("gendered_noun_words.csv", index=False)

In [ ]:
#save to csv file after preprocessing
df_gendered_word_no_noun.to_csv("gendered_no_noun_words.csv", index=False)

In [ ]:
replace_tolist(r"gendered_noun_words.csv")

'gendered_noun_words.csv'

In [ ]:
replace_tolist(r"gendered_no_noun_words.csv")

'gendered_no_noun_words.csv'

## Unconscious Bias

In [ ]:
df_ub = df_rules[df_rules["Category"]=='unconscious_bias']

In [ ]:
#info unc_bias
df_ub.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 751 entries, 1487 to 2286
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Lemma                  751 non-null    object
 1   Word_Type              751 non-null    object
 2   Category               751 non-null    object
 3   Primary_subcategory    751 non-null    object
 4   Secondary_subcategory  174 non-null    object
 5   Sg_all_split           751 non-null    object
 6   Pl_all                 751 non-null    object
 7   Pl_all_split           751 non-null    object
 8   Alt_split              751 non-null    object
 9   False_Positives        33 non-null     object
dtypes: object(10)
memory usage: 64.5+ KB


In [ ]:
#check missing values in Secondary subcategory
df_ub[df_ub['Secondary_subcategory'].notnull()]

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives
1581,abort,v,unconscious_bias,agentic,tech,"[terminate, stop, cancel, pause, suspend]",,[],"[terminate, stop, cancel, pause, suspend]",NaN
1582,achiever,s,unconscious_bias,agentic,competitive,"[person who sees things through(...), person w...",people who see things through(...) | those wit...,"[people who see things through(...), those wit...","[person who sees things through(...), person w...",NaN
1583,adventurous,a,unconscious_bias,agentic,competitive,"[enterprising, resourceful, happy to explore n...",,[],"[enterprising, resourceful, happy to explore n...",NaN
1584,aggression,s,unconscious_bias,agentic,hard_hearted,"[commitment to achieving a goal, focus on purp...",positive attitudes toward using one's competen...,[positive attitudes toward using one's compete...,"[commitment to achieving a goal, focus on purp...",NaN
1585,aggressive,a,unconscious_bias,agentic,hard_hearted,"[energetic, emphatic, invested, spirited, open...",,[],"[energetic, emphatic, invested, spirited, open...",NaN
1586,alone,a,unconscious_bias,agentic,competitive,"[uninterrupted, without distractions, --- don'...",--- don't use for people,[--- don't use for people],"[--- don't use for people, uninterrupted, with...",NaN
1587,ambition,s,unconscious_bias,agentic,competitive,"[growth mindset, dedication, willingness to go...",growth mindset | dedication | willingness to g...,"[growth mindset, dedication, willingness to go...","[growth mindset, dedication, willingness to go...",NaN
1588,ambitious,a,unconscious_bias,agentic,competitive,"[growth-minded, keen and enthusiastic, --- don...",--- don't use for people,[--- don't use for people],"[--- don't use for people, growth-minded, keen...",NaN
1589,analysis,s,unconscious_bias,agentic,rough,"[investigation, study with scientific care, cl...",interpretation of data | data explorations | d...,"[interpretation of data, data explorations, di...","[investigation, study with scientific care, cl...",analysis team | competitive analysis | data an...
1590,analytical,a,unconscious_bias,agentic,rough,"[logical, systematical, investigative, explora...",,[],"[logical, systematical, investigative, explora...",NaN


In [ ]:
#fill the missing values in "Secondary subcategory" by values from "Primary_Category"
df_ub.Secondary_subcategory.fillna(df_ub.Primary_subcategory, inplace=True)

<ipython-input-181-cfde6227fba8>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ub.Secondary_subcategory.fillna(df_ub.Primary_subcategory, inplace=True)


In [ ]:
df_ub.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 751 entries, 1487 to 2286
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Lemma                  751 non-null    object
 1   Word_Type              751 non-null    object
 2   Category               751 non-null    object
 3   Primary_subcategory    751 non-null    object
 4   Secondary_subcategory  751 non-null    object
 5   Sg_all_split           751 non-null    object
 6   Pl_all                 751 non-null    object
 7   Pl_all_split           751 non-null    object
 8   Alt_split              751 non-null    object
 9   False_Positives        33 non-null     object
dtypes: object(10)
memory usage: 64.5+ KB


In [ ]:
df_ub.query("Lemma=='go the extra mile'")

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives
1651,go the extra mile,~v||a|~s,unconscious_bias,agentic,competitive,"[think ahead, make a special effort, make cust...",,[],"[think ahead, make a special effort, make cust...",NaN


In [ ]:
df_ub[df_ub["Word_Type"]=='-s']

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives


In [ ]:
df_ub[df_ub["Word_Type"]=='~s']

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives
1491,Asperger,~s,unconscious_bias,advanced_ability,advanced_ability,"[Autistic, neurodivergent, on the autism spect...","--- only if the person prefers the term, or wh...","[--- only if the person prefers the term, or w...","[--- only if the person prefers the term, or w...",NaN
1553,gramps,~s,unconscious_bias,age_old,age_old,"[grandy, grandparent, older person, grandies, ...",grandies | grandparents | older people,"[grandies, grandparents, older people]","[grandy, grandparent, older person, grandies, ...",NaN
1855,IQ-test,~s,unconscious_bias,cognitive_perception,cognitive_perception,"[aptitude test, deliberative test, learning sk...",aptitude tests | deliberative tests | learning...,"[aptitude tests, deliberative tests, learning ...","[aptitude test, deliberative test, learning sk...",NaN
1923,gals,~s,unconscious_bias,gender_identity,gender_identity,"[you, folks, people, everyone, friends]",,[],"[you, folks, people, everyone, friends]",NaN
2160,grandfathering,~s,unconscious_bias,racist_source,racist_source,"[exempting, making an exception for]",,[],"[exempting, making an exception for]",NaN


In [ ]:
df_ub[df_ub["Word_Type"]=='=s']

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives


In [ ]:
#filter datframe to have plural for subj(noun) without special caracters and empty strings
df_ub_word_pl = df_ub[(df_ub["Pl_all"]!='')&(df_ub["Pl_all"]!='-')&(df_ub["Word_Type"]=='s')]

In [ ]:
df_ub_word_pl.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 136 entries, 1494 to 2272
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Lemma                  136 non-null    object
 1   Word_Type              136 non-null    object
 2   Category               136 non-null    object
 3   Primary_subcategory    136 non-null    object
 4   Secondary_subcategory  136 non-null    object
 5   Sg_all_split           136 non-null    object
 6   Pl_all                 136 non-null    object
 7   Pl_all_split           136 non-null    object
 8   Alt_split              136 non-null    object
 9   False_Positives        16 non-null     object
dtypes: object(10)
memory usage: 11.7+ KB


#### save the left data into separate dataframe

In [ ]:
plural_list = list(df_ub_word_pl["Lemma"])

In [ ]:
df_ub_word_left = df_ub.query("Lemma != @plural_list")

In [ ]:
df_ub_word_left.query("Lemma=='go the extra mile'")

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all_split,Alt_split,False_Positives
1651,go the extra mile,~v||a|~s,unconscious_bias,agentic,competitive,"[think ahead, make a special effort, make cust...",[],"[think ahead, make a special effort, make cust...",NaN


#### remove unnecessary column "Pl_all"

In [ ]:
df_ub_word_left = df_ub_word_left.drop(labels=["Pl_all"], axis=1)
df_ub_word_pl = df_ub_word_pl.drop(labels=["Pl_all"], axis=1)


In [ ]:
df_ub_word_left.query("Lemma=='go the extra mile'")

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all_split,Alt_split,False_Positives
1651,go the extra mile,~v||a|~s,unconscious_bias,agentic,competitive,"[think ahead, make a special effort, make cust...",[],"[think ahead, make a special effort, make cust...",NaN


In [ ]:
# replace secondary_subcategory by primary_subcategory for ub_noun because it not used by the moment
df_ub_word_pl.Secondary_subcategory=df_ub_word_pl.Primary_subcategory

In [ ]:
df_ub_word_pl.head()

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all_split,Alt_split,False_Positives
1494,birth-defect,s,unconscious_bias,advanced_ability,advanced_ability,"[is disabled, has a disability, congenital dis...","[are disabled, having disabilities, congenital...",[---name congenital health condition or disabi...,NaN
1501,diffability,s,unconscious_bias,advanced_ability,advanced_ability,"[disability, ability status, physical disabili...","[disabilities, ability status, physical disabi...","[disability, ability status, physical disabili...",NaN
1507,dotard,s,unconscious_bias,advanced_ability,advanced_ability,"[person in their dotage(...), physically weak ...","[persons in their dotage(...), physically weak...","[person in their dotage(...), physically weak ...",NaN
1517,inmate,s,unconscious_bias,advanced_ability,advanced_ability,"[patient in a psychiatric hospital ward(...), ...","[patients in a psychiatric hospital unit(...),...","[patient in a psychiatric hospital ward(...), ...",NaN
1535,symptom,s,unconscious_bias,advanced_ability,advanced_ability,"[trait, quality, how ((health condition or dis...","[traits, qualities, how ((health condition or ...","[trait, quality, how ((health condition or dis...",NaN


#### save to csv files after preprocessing

In [ ]:
#save to csv files after preprocessing
df_ub_word_left.to_csv("ub_no_plur_words.csv", index=False)
df_ub_word_pl.to_csv("ub_plur_words.csv", index=False)


In [ ]:
replace_tolist(r"ub_no_plur_words.csv")

'ub_no_plur_words.csv'

In [ ]:
replace_tolist(r"ub_plur_words.csv")

'ub_plur_words.csv'